В цьому домашньому завданні ми знову працюємо з даними з нашого змагання ["Bank Customer Churn Prediction (DLU Course)"](https://www.kaggle.com/t/7c080c5d8ec64364a93cf4e8f880b6a0).

Тут ми побудуємо рішення задачі класифікації з використанням kNearestNeighboors, знайдемо оптимальні гіперпараметри для цього методу і зробимо базові ансамблі. Це дасть змогу порівняти перформанс моделі з попередніми вивченими методами.

0. Зчитайте дані `train.csv` та зробіть препроцесинг використовуючи написаний Вами скрипт `process_bank_churn.py` так, аби в результаті отримати дані в розбитті X_train, train_targets, X_val, val_targets для експериментів.

  Якщо Вам не вдалось реалізувати в завданні `2.3. Дерева прийняття рішень` скрипт `process_bank_churn.py` - можна скористатись готовим скриптом з запропонованого рішення того завдання.

In [7]:
!wget -O process_bank_churn.py "https://raw.githubusercontent.com/mariyagryn/ml_course_projects/refs/heads/main/Maching%20Learning/Module%202/process_bank_churn.py"

--2026-08-10 10:00:24--  https://raw.githubusercontent.com/mariyagryn/ml_course_projects/refs/heads/main/Maching%20Learning/Module%202/process_bank_churn.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9771 (9.5K) [text/plain]
Saving to: ‘process_bank_churn.py’

process_bank_churn. 100%[===================>]   9.54K  --.-KB/s    in 0.001s  

2026-08-10 10:00:25 (14.6 MB/s) - ‘process_bank_churn.py’ saved [9771/9771]



In [27]:
import pandas as pd
import numpy as np

from process_bank_churn import preprocess_data
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV

In [11]:
train_df = pd.read_csv("train.csv")

X_train, train_targets, X_val, val_targets, input_cols, scaler, encoder = preprocess_data(
    train_df
)

In [9]:
X_train.shape, train_targets.shape, X_val.shape, val_targets.shape

((11250, 14), (11250,), (3750, 14), (3750,))

1. Навчіть на цих даних класифікатор kNN з параметрами за замовченням і виміряйте точність з допомогою AUROC на тренувальному та валідаційному наборах. Зробіть заключення про отриману модель: вона хороша/погана, чи є high bias/high variance?

In [12]:
knn = KNeighborsClassifier()
knn.fit(X_train, train_targets)

train_probs = knn.predict_proba(X_train)[:, 1]
val_probs = knn.predict_proba(X_val)[:, 1]

train_auc = roc_auc_score(train_targets, train_probs)
val_auc = roc_auc_score(val_targets, val_probs)

print(f"Train AUROC: {train_auc:.4f}")
print(f"Validation AUROC: {val_auc:.4f}")

Train AUROC: 0.9496
Validation AUROC: 0.8534


Модель має хороший результат на обох вибірках: Train AUROC = 0.9496, Validation AUROC = 0.8534.

Спостерігається помітна різниця між результатами на тренувальній та валідаційній вибірках, що свідчить про певний рівень перенавчання моделі (high variance).

При цьому модель не має high bias, оскільки AUROC на валідаційній вибірці залишається досить високим.

2. Використовуючи `GridSearchCV` знайдіть оптимальне значення параметра `n_neighbors` для класифікатора `kNN`. Псотавте крос валідацію на 5 фолдів.

  Після успішного завершення пошуку оптимального гіперпараметра
    - виведіть найкраще значення параметра
    - збережіть в окрему змінну `knn_best` найкращу модель, знайдену з `GridSearchCV`
    - оцініть якість передбачень  `knn_best` на тренувальній і валідаційній вибірці з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи стала вона краще порівняно з попереднім пукнтом (2) цього завдання? Чи є вона краще за дерево прийняття рішень з попереднього ДЗ?

In [14]:
knn = KNeighborsClassifier()

param_grid = {
    "n_neighbors": range(1, 31)
}

grid_search = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

grid_search.fit(X_train, train_targets)

print("Best n_neighbors:", grid_search.best_params_["n_neighbors"])
print("Best CV AUROC:", grid_search.best_score_)

Best n_neighbors: 17
Best CV AUROC: 0.8688957935134323


In [15]:
knn_best = grid_search.best_estimator_

In [16]:
train_probs_best = knn_best.predict_proba(X_train)[:, 1]
val_probs_best = knn_best.predict_proba(X_val)[:, 1]

train_auc_best = roc_auc_score(train_targets, train_probs_best)
val_auc_best = roc_auc_score(val_targets, val_probs_best)

print(f"Train AUROC: {train_auc_best:.4f}")
print(f"Validation AUROC: {val_auc_best:.4f}")

Train AUROC: 0.9184
Validation AUROC: 0.8798


Порівняно з kNN з параметрами за замовчуванням модель стала кращою: Validation AUROC зріс з 0.8534 до 0.8798. Також зменшилася різниця між Train та Validation AUROC, що свідчить про зменшення перенавчання та variance.

Отже, оптимізована модель має хорошу якість і краще узагальнює на нових даних.

Decision Tree показала кращий результат на валідаційній вибірці, ніж оптимізований kNN. Різниця становить приблизно 0.0453 AUROC на користь дерева.

3. Виконайте пошук оптимальних гіперпараметрів для `DecisionTreeClassifier` з `GridSearchCV` за сіткою параметрів
  - `max_depth` від 1 до 20 з кроком 2
  - `max_leaf_nodes` від 2 до 10 з кроком 1

  Обовʼязково при цьому ініціюйте модель з фіксацією `random_state`.

  Поставте кросвалідацію на 3 фолди, `scoring='roc_auc'`, та виміряйте, скільки часу потребує пошук оптимальних гіперпараметрів.

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення параметра
    - збережіть в окрему змінну `dt_best` найкращу модель, знайдену з `GridSearchCV`
    - оцініть якість передбачень  `dt_best` на тренувальній і валідаційній вибірці з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи ця модель краща за ту, що ви знайшли вручну?

In [19]:
import time
from sklearn.tree import DecisionTreeClassifier

In [21]:
dt = DecisionTreeClassifier(random_state=42)
param_grid = {
    "max_depth": range(1, 21, 2),
    "max_leaf_nodes": range(2, 11)
}

grid_search_dt = GridSearchCV(
    estimator=dt,
    param_grid=param_grid,
    cv=3,
    scoring="roc_auc",
    n_jobs=-1
)

start_time = time.time()

grid_search_dt.fit(X_train, train_targets)

end_time = time.time()

search_time = end_time - start_time

print(f"Search time: {search_time:.2f} seconds")
print("Best parameters:", grid_search_dt.best_params_)
print("Best CV AUROC:", grid_search_dt.best_score_)

Search time: 6.47 seconds
Best parameters: {'max_depth': 5, 'max_leaf_nodes': 10}
Best CV AUROC: 0.896755388100145


In [22]:
dt_best = grid_search_dt.best_estimator_

In [23]:
print("Best max_depth:", grid_search_dt.best_params_["max_depth"])
print("Best max_leaf_nodes:", grid_search_dt.best_params_["max_leaf_nodes"])

Best max_depth: 5
Best max_leaf_nodes: 10


In [24]:
train_probs_dt = dt_best.predict_proba(X_train)[:, 1]
val_probs_dt = dt_best.predict_proba(X_val)[:, 1]

train_auc_dt = roc_auc_score(train_targets, train_probs_dt)
val_auc_dt = roc_auc_score(val_targets, val_probs_dt)

print(f"Train AUROC: {train_auc_dt:.4f}")
print(f"Validation AUROC: {val_auc_dt:.4f}")

Train AUROC: 0.9010
Validation AUROC: 0.9018


Ознак high variance / сильного перенавчання немає. Ця модель краща за попередню, але не краща за ту що ми знайшли вручну. Вручну ми досягли AUROC 0,93, а зараз лише 0,90

4. Виконайте пошук оптимальних гіперпараметрів для `DecisionTreeClassifier` з `RandomizedSearchCV` за заданою сіткою параметрів і кількість ітерацій 40.

  Поставте кросвалідацію на 3 фолди, `scoring='roc_auc'`, зафіксуйте `random_seed` процедури крос валідації та виміряйте, скільки часу потребує пошук оптимальних гіперпараметрів.

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення параметра
    - збережіть в окрему змінну `dt_random_search_best` найкращу модель, знайдену з `RandomizedSearchCV`
    - оцініть якість передбачень  `dt_random_search_best` на тренувальній і валідаційній вибірці з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи ця модель краща за ту, що ви знайшли з `GridSearch`?
    - проаналізуйте параметри `dt_random_search_best` і порівняйте з параметрами `dt_best` - яку бачите відмінність? Ця вправа потрібна аби зрозуміти, як різні налаштування `DecisionTreeClassifier` впливають на якість моделі.

In [28]:
params_dt = {
    'criterion': ['gini', 'entropy'],
    'splitter': ['best', 'random'],
    'max_depth': np.arange(1, 20),
    'max_leaf_nodes': np.arange(2, 20),
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': [None, 'sqrt', 'log2']
}

In [30]:
from sklearn.model_selection import RandomizedSearchCV

In [31]:
dt_random = DecisionTreeClassifier(random_state=42)

random_search_dt = RandomizedSearchCV(
    estimator=dt_random,
    param_distributions=params_dt,
    n_iter=40,
    cv=3,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1
)

start_time = time.time()

random_search_dt.fit(X_train, train_targets)

end_time = time.time()

search_time_random = end_time - start_time

print(f"Search time: {search_time_random:.2f} seconds")
print("Best parameters:", random_search_dt.best_params_)
print("Best CV AUROC:", random_search_dt.best_score_)

Search time: 3.43 seconds
Best parameters: {'splitter': 'best', 'min_samples_split': 20, 'min_samples_leaf': 2, 'max_leaf_nodes': np.int64(14), 'max_features': None, 'max_depth': np.int64(16), 'criterion': 'entropy'}
Best CV AUROC: 0.9118622813318175


In [32]:
dt_random_search_best = random_search_dt.best_estimator_

In [33]:
print("Best parameters:")
for param, value in random_search_dt.best_params_.items():
    print(f"{param}: {value}")

Best parameters:
splitter: best
min_samples_split: 20
min_samples_leaf: 2
max_leaf_nodes: 14
max_features: None
max_depth: 16
criterion: entropy


In [34]:
train_probs_random = dt_random_search_best.predict_proba(X_train)[:, 1]
val_probs_random = dt_random_search_best.predict_proba(X_val)[:, 1]

train_auc_random = roc_auc_score(train_targets, train_probs_random)
val_auc_random = roc_auc_score(val_targets, val_probs_random)

print(f"Train AUROC: {train_auc_random:.4f}")
print(f"Validation AUROC: {val_auc_random:.4f}")

Train AUROC: 0.9166
Validation AUROC: 0.9176


Отримана модель є кращою за модель, знайдену за допомогою GridSearchCV, оскільки Validation AUROC зріс з 0.9018 до 0.9176. При цьому різниця між Train та Validation AUROC дуже мала, тому модель добре узагальнює дані та не має значного перенавчання.

5. Якщо у Вас вийшла метрика `AUROC` в цій серії експериментів - зробіть ще один `submission` на Kaggle і додайте код для цього і скріншот скора на публічному лідерборді нижче.

  Сподіваюсь на цьому етапі ви вже відчули себе справжнім дослідником 😉

In [39]:
test_df = pd.read_csv("test.csv")
from process_bank_churn import preprocess_new_data

X_test = preprocess_new_data(
    test_df,
    input_cols=input_cols,
    scaler=scaler,
    encoder=encoder
)

test_probs = dt_random_search_best.predict_proba(X_test)[:, 1]
submission = pd.DataFrame({
    "id": test_df["id"],
    "Exited": test_probs
})

submission.to_csv("submission.csv", index=False)

submission.head()

,id,Exited
0,15000,0.231405
1,15001,0.012522
2,15002,0.202765
3,15003,0.564065
4,15004,0.080196
